<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Fine_Tuning_with_LoRA_and_QLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning an Open Model Locally with LoRA and QLoRA

In this notebook we fine-tune `gpt-oss-20b`, OpenAI's open-weight reasoning model, on a free Google Colab T4 GPU. We use QLoRA: the 21B-parameter base model stays frozen in 4-bit, and we train only a small set of LoRA adapter matrices on top of it.

**Before you run anything**, switch the runtime to a GPU: *Runtime > Change runtime type > T4 GPU*. The notebook will not run on a CPU runtime.

Expect the install cell to take a few minutes, the model download about 13 GB, and the training run roughly 15 minutes at the default settings.

## Install Packages and Setup Variables

In [ ]:
%%capture
import importlib.util
import os

# Package versions match Unsloth's gpt-oss Colab recipe as of August 2026.
!pip install --upgrade -qqq uv

if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy, _pil = "numpy", "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base]==2026.8.7" "unsloth[base]==2026.8.10" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels

!uv pip install --upgrade --no-deps \
    "transformers==4.56.2" "tokenizers>=0.22.0,<=0.23.0" "trl==0.22.2" \
    "unsloth==2026.8.10" "unsloth_zoo==2026.8.7"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

Check that a GPU is actually attached and see how much VRAM you were given. A free Colab T4 reports about 15 GB.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found. Use Runtime > Change runtime type > T4 GPU."

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"Total VRAM: {round(gpu.total_memory / 1024 ** 3, 2)} GB")

## Load gpt-oss-20b in 4-Bit

`FastLanguageModel.from_pretrained` downloads the model and quantizes the frozen base weights to 4-bit. `max_seq_length` caps how long a single training example can be, and short sequences are what keep this run inside a T4's memory budget.

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gpt-oss-20b",
    dtype=None,             # None lets Unsloth pick the right dtype for the GPU
    max_seq_length=max_seq_length,
    load_in_4bit=True,      # the QLoRA part: frozen base weights in 4-bit
    full_finetuning=False,
    # token = "hf_...",     # only needed for gated models
)

Before training anything, let's see what the base model already does. `gpt-oss` exposes a `reasoning_effort` setting (`"low"`, `"medium"`, `"high"`) that controls how many thinking tokens it spends before answering.

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    reasoning_effort="low",
).to("cuda")

_ = model.generate(**inputs, max_new_tokens=64, streamer=TextStreamer(tokenizer))

## Attach LoRA Adapters

`get_peft_model` freezes the base model and inserts a trainable low-rank pair (A and B) next to each of the listed projection matrices. `r` sets the rank of that pair, and `lora_alpha` scales how strongly the adapter's output is added back into the layer.

We use `r=16` with `lora_alpha=16`, the starting pair Unsloth recommends. Drop back to `r=8` if the runtime runs out of memory.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                      # rank: capacity of the adapter (8, 16, 32, 64, 128)
    lora_alpha=16,             # scaling factor applied to the adapter output
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",     # attention projections
        "gate_proj", "up_proj", "down_proj",        # feed-forward projections
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Count what we are actually training. This is the number that makes local fine-tuning possible on a 15 GB card.

In [ ]:
model.print_trainable_parameters()

## Prepare the Training Data

We use `HuggingFaceH4/Multilingual-Thinking`, a small set of chain-of-thought examples where the reasoning is written in French, Spanish, German, or Italian. It is a convenient teaching target because the behavior we are training is easy to check by eye: after fine-tuning, the model should think in the requested language.

Each row is a list of chat messages. `standardize_sharegpt` normalizes the column names, and `formatting_prompts_func` renders each conversation into a single `text` string using the model's own chat template.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt


def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}


dataset = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train")
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched=True)

print(dataset)

Print one formatted example. `gpt-oss` uses OpenAI's Harmony format, so you will see channel markers such as `<|start|>`, `<|channel|>`, and `<|message|>`, with the reasoning and the final answer on separate channels.

In [ ]:
print(dataset[0]["text"][:1500])

### Using Your Own Data

Your own dataset needs exactly one thing: a `messages` list per row, with `role` and `content` keys. Write it as JSONL, load it with the `json` loader, and run it through the same formatting function. The cell below builds a three-row example so you can see the shape.

In [ ]:
import json

my_examples = [
    {
        "messages": [
            {"role": "system", "content": "You answer questions about the pandas library. Be brief and show code."},
            {"role": "user", "content": "How do I drop rows that contain missing values?"},
            {"role": "assistant", "content": "Use `df.dropna()`. Pass `subset=['col']` to only look at specific columns."},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You answer questions about the pandas library. Be brief and show code."},
            {"role": "user", "content": "How do I read a CSV with a semicolon separator?"},
            {"role": "assistant", "content": "Use `pd.read_csv(path, sep=';')`."},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You answer questions about the pandas library. Be brief and show code."},
            {"role": "user", "content": "How do I count unique values in a column?"},
            {"role": "assistant", "content": "Use `df['col'].value_counts()`, or `df['col'].nunique()` for just the count."},
        ]
    },
]

with open("my_dataset.jsonl", "w") as f:
    for row in my_examples:
        f.write(json.dumps(row) + "\n")

my_dataset = load_dataset("json", data_files="my_dataset.jsonl", split="train")
my_dataset = my_dataset.map(formatting_prompts_func, batched=True)

print(my_dataset[0]["text"][:800])

Your rows are already in `messages` form, so `standardize_sharegpt` is unnecessary here. To train on this data instead, replace `train_dataset=dataset` with `train_dataset=my_dataset` in the trainer below. Three rows will not teach the model anything, so build a few hundred before you expect a real change in behavior.

## Train the Adapters

`SFTTrainer` runs supervised fine-tuning on the `text` column. The settings below are sized for a T4: a batch of one sequence at a time, with gradients accumulated over four steps so the effective batch size is four.

The learning rate of `1e-4` assumes a full run of one to three epochs, which is what a fine-tune you would ship looks like. `max_steps=30` caps this notebook at 30 optimizer steps instead, so the demo fits one free Colab T4 session. For a real run, set `num_train_epochs=1` and `max_steps=None`, and raise the steps or epochs rather than the learning rate.

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=30,               # num_train_epochs=1 and max_steps=None for a full run
        learning_rate=1e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

By default the trainer computes loss over the whole conversation, including the user's question. We only want the model to learn how to *answer*, so we mask everything before the assistant's turn.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start|>user<|message|>",
    response_part="<|start|>assistant<|channel|>final<|message|>",
)

Let's confirm the masking worked. The first output is the full example. The second replaces every masked token with padding, so only the part the model is scored on remains.

In [ ]:
print(tokenizer.decode(trainer.train_dataset[100]["input_ids"])[:800])

In [ ]:
labels = trainer.train_dataset[100]["labels"]
masked = [tokenizer.pad_token_id if x == -100 else x for x in labels]
print(tokenizer.decode(masked).replace(tokenizer.pad_token, " ")[:800])

Record memory before training so we can measure what the adapters cost.

In [ ]:
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 ** 3, 3)
max_memory = round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 3)
print(f"{start_gpu_memory} GB reserved of {max_memory} GB total")

Now run the training. The loss printed at each step is the value we read in the next section.

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 ** 3, 3)
print(f"Training took {round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes.")
print(f"Peak reserved memory: {used_memory} GB ({round(used_memory / max_memory * 100, 1)}% of the card)")
print(f"Added by training: {round(used_memory - start_gpu_memory, 3)} GB")

## Read the Loss Curve

`logging_steps=1` recorded the loss at every optimizer step, so the run history is a list we can plot directly.

In [ ]:
import matplotlib.pyplot as plt

history = [h for h in trainer.state.log_history if "loss" in h]
steps = [h["step"] for h in history]
losses = [h["loss"] for h in history]

plt.figure(figsize=(7, 4))
plt.plot(steps, losses, marker="o", linewidth=1)
plt.xlabel("Training step")
plt.ylabel("Training loss")
plt.title("Training loss per step")
plt.grid(alpha=0.3)
plt.show()

print(f"First logged loss: {losses[0]:.4f}")
print(f"Last logged loss:  {losses[-1]:.4f}")

With an effective batch size of four, the curve is noisy: single steps jump around depending on which examples landed in the batch. Read the trend rather than any one point. A curve that drifts down and then flattens means the adapters have absorbed what this dataset has to teach, and more steps will mostly memorize.

## Run the Fine-Tuned Model

The adapters are live on the model object, so we can generate straight away. We ask for reasoning in French, which is one of the behaviors the dataset teaches.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "reasoning language: French\n\nYou are a helpful assistant that can solve mathematical problems.",
    },
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    reasoning_effort="medium",
).to("cuda")

_ = model.generate(**inputs, max_new_tokens=256, streamer=TextStreamer(tokenizer))

## Save and Export

`save_pretrained` writes only the adapter weights, which is a few tens of megabytes rather than a full model copy. Reloading means loading the base model again and applying the adapter on top.

In [ ]:
model.save_pretrained("gpt_oss_lora")
tokenizer.save_pretrained("gpt_oss_lora")

# model.push_to_hub("your-username/gpt_oss_lora", token="hf_...")

To reload the adapters in a fresh runtime, point `from_pretrained` at the saved directory. Set the flag to `True` when you want to run it.

In [ ]:
if False:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="gpt_oss_lora",
        max_seq_length=1024,
        dtype=None,
        load_in_4bit=True,
    )

To serve the model anywhere else, merge the adapters into the base weights. `mxfp4` keeps the native 4-bit format and is the smaller, faster option, while `merged_16bit` produces a standard float16 checkpoint that any inference stack can read.

Merging needs far more disk and RAM than a T4 session comfortably has, so these are switched off by default. Run them on a larger runtime.

In [ ]:
if False:
    model.save_pretrained_merged("gpt_oss_merged_mxfp4", tokenizer, save_method="mxfp4")

if False:
    model.save_pretrained_merged("gpt_oss_merged_16bit", tokenizer, save_method="merged_16bit")

if False:
    model.push_to_hub_merged(
        "your-username/gpt_oss_merged_mxfp4",
        tokenizer,
        save_method="mxfp4",
        token="hf_...",
    )

## What to Try Next

- **Change the rank.** Re-run with `r=8` or `r=32` and compare the final loss and the peak memory figure. Higher rank buys capacity and costs VRAM.
- **Swap in your own data.** Build a few hundred rows in the JSONL shape above, and train on a behavior you can check by eye.
- **Hold out a validation set.** Split off ten percent of your rows and score them after training, so you can tell learning apart from memorization.
- **Merge and serve.** Export with `merged_16bit` on a larger GPU runtime, then run the result through a standard inference server.